In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
X = np.load("../cache/text_concat.npy")


In [3]:
X[0].size
train_tokens = X[0:9988]
X_train = torch.from_numpy(train_tokens)
dev_tokens = X[9988:11096]
X_val = torch.from_numpy(dev_tokens)
test_tokens = X[11096:137056]
X_test = torch.from_numpy(test_tokens)

#0 - 9987 is train
#9988 - 11095 is dev
#11096 - 13705 is test

In [4]:
train = pd.read_csv('../data/labels/train_sent_emo.csv')
manifest = pd.read_csv('../data/manifest.csv')

In [5]:
EMOTIONS = {"neutral":0, "joy":1, "surprise":2, "anger":3, "sadness":4, "disgust":5, "fear":6}


In [6]:
manifest['emo_encoded'] = manifest["Emotion"].map(EMOTIONS)

In [7]:
Y_train = manifest[manifest.split == 'train'].emo_encoded.to_numpy()
Y_train = torch.from_numpy(Y_train)

Y_val = manifest[manifest.split == 'dev'].emo_encoded.to_numpy()
Y_val = torch.from_numpy(Y_val)

Y_test = manifest[manifest.split == 'test'].emo_encoded.to_numpy()
Y_test = torch.from_numpy(Y_test)

In [8]:
class EmotionClassifier(nn.Module):

    def __init__(self):
        super().__init__()


        self.network = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(128, len(EMOTIONS))
        )

    def forward(self, x):
        return self.network(x)


counts = torch.bincount(Y_train)
class_weights = (1.0 / counts.float().sqrt())
class_weights = class_weights / class_weights.sum() * len(EMOTIONS)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)




In [9]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score


model = EmotionClassifier()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

train_ds = TensorDataset(X_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

best_val_loss = -1
patience = 50
patience_counter = 0
best_weights = {k: v.clone() for k, v in model.state_dict().items()}


best_val_loss, best_weights, stale = float("inf"), None, 0
patience = 10

for epoch in range(100):
    model.train()
    for xb, yb in train_dl:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), Y_val).item()
        val_f1 = f1_score(Y_val, model(X_val).argmax(1), average="weighted")

    if val_loss < best_val_loss:
        best_val_loss, stale = val_loss, 0
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        flag = " *"
    else:
        stale += 1
        flag = ""
    print(f"epoch {epoch:3d}  val_loss {val_loss:.4f}  dev_wF1 {val_f1:.4f}{flag}")
    if stale >= patience:
        break

model.load_state_dict(best_weights)

epoch   0  val_loss 1.9241  dev_wF1 0.1766 *
epoch   1  val_loss 1.8093  dev_wF1 0.3817 *
epoch   2  val_loss 1.7697  dev_wF1 0.4059 *
epoch   3  val_loss 1.7446  dev_wF1 0.4199 *
epoch   4  val_loss 1.7160  dev_wF1 0.4351 *
epoch   5  val_loss 1.7007  dev_wF1 0.4296 *
epoch   6  val_loss 1.6894  dev_wF1 0.4337 *
epoch   7  val_loss 1.6804  dev_wF1 0.4391 *
epoch   8  val_loss 1.6594  dev_wF1 0.4427 *
epoch   9  val_loss 1.6547  dev_wF1 0.4385 *
epoch  10  val_loss 1.6528  dev_wF1 0.4407 *
epoch  11  val_loss 1.6408  dev_wF1 0.4479 *
epoch  12  val_loss 1.6438  dev_wF1 0.4442
epoch  13  val_loss 1.6398  dev_wF1 0.4420 *
epoch  14  val_loss 1.6297  dev_wF1 0.4526 *
epoch  15  val_loss 1.6260  dev_wF1 0.4506 *
epoch  16  val_loss 1.6213  dev_wF1 0.4527 *
epoch  17  val_loss 1.6214  dev_wF1 0.4522
epoch  18  val_loss 1.6233  dev_wF1 0.4463
epoch  19  val_loss 1.6097  dev_wF1 0.4462 *
epoch  20  val_loss 1.6064  dev_wF1 0.4586 *
epoch  21  val_loss 1.6024  dev_wF1 0.4418 *
epoch  22  val_l

<All keys matched successfully>

In [10]:

model.eval()
with torch.no_grad():
    tr_pred = model(X_train).argmax(1)
    va_pred = model(X_val).argmax(1)

print("train wF1", f1_score(Y_train, tr_pred, average="weighted"))
print("dev   wF1", f1_score(Y_val, va_pred, average="weighted"))

train wF1 0.6112986777216985
dev   wF1 0.46067276067611607


In [11]:
from sklearn.metrics import classification_report
print(classification_report(Y_val, va_pred, target_names=EMOTIONS, zero_division=0))

              precision    recall  f1-score   support

     neutral       0.61      0.57      0.59       469
         joy       0.38      0.37      0.37       163
    surprise       0.36      0.46      0.40       150
       anger       0.37      0.42      0.39       153
     sadness       0.42      0.32      0.37       111
     disgust       0.12      0.14      0.12        22
        fear       0.21      0.17      0.19        40

    accuracy                           0.46      1108
   macro avg       0.35      0.35      0.35      1108
weighted avg       0.47      0.46      0.46      1108



In [12]:
model.eval()
with torch.no_grad():
    print("initial val_loss:", loss_fn(model(X_val), Y_val).item())

initial val_loss: 1.550566554069519
